# Neu — проба Colab

Что нажать: **Среда выполнения → Выполнить все** (Runtime → Run all). Когда Colab попросит доступ к Google Диску — разрешить.

Ноутбук запишет описание машины в папку **Neu_colab/probe** на вашем Диске. Claude прочтёт файл оттуда через подключённый Google Диск и закоммитит в репозиторий. Личных данных ноутбук не пишет.


In [ ]:
# 1. Описание машины
import json, os, platform, subprocess, sys, time
t0 = time.time()
info = {'python': sys.version.split()[0], 'platform': platform.platform(), 'cpus': os.cpu_count()}
import numpy as np
info['numpy'] = np.__version__
a = np.random.default_rng(0).standard_normal((2000, 2000))
t = time.time(); _ = a @ a; info['matmul_2000_s'] = round(time.time() - t, 3)
try:
    info['nvidia_smi'] = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=20).stdout.strip() or 'нет'
except Exception:
    info['nvidia_smi'] = 'нет'
try:
    import torch
    info['torch'] = torch.__version__; info['cuda'] = torch.cuda.is_available()
except Exception as e:
    info['torch'] = 'нет'
info['seconds'] = round(time.time() - t0, 2)
info['utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
print(json.dumps(info, ensure_ascii=False, indent=1))


In [ ]:
# 2. Запись результата на Google Диск (Colab попросит разрешение)
from google.colab import drive
drive.mount('/content/drive')
out = '/content/drive/MyDrive/Neu_colab/probe'
os.makedirs(out, exist_ok=True)
json.dump(info, open(os.path.join(out, 'probe.json'), 'w'), ensure_ascii=False, indent=1)
drive.flush_and_unmount()
print('записано в Neu_colab/probe/probe.json — можно написать Claude «готово»')
